# Construction de graphe de citations

Dans cette partie nous allons mettre en place un graphe de citations et étudier ses caractéristiques.

## Importations des librairies nécessaires

In [4]:
import json
import numpy as np
import networkx as nx
from collections import defaultdict, Counter
import os
import pickle
from sklearn.feature_extraction.text import CountVectorizer

import warnings

warnings.filterwarnings("ignore")

# For embeddings and similarity computation
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity

    print("Required libraries imported successfully!")
except ImportError as e:
    print(f"Missing library: {e}")
    print(
        "Please install with: pip install sentence-transformers scikit-learn networkx"
    )

Required libraries imported successfully!


## Chargement des corpus

In [5]:
def load_corpus(file_path: str) -> dict[str, dict]:
    """
    TODO

    Load corpus data from JSONL file.
    Returns dictionary mapping document IDs to document data.
    """
    corpus = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            obj = json.loads(line)
            docid = str(obj["_id"])
            corpus[docid] = obj
    return corpus


def load_queries(file_path: str) -> dict[str, dict]:
    """
    TODO

    Load query data from JSONL file.
    Returns dictionary mapping query IDs to query data.
    """
    queries = {}
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            obj = json.loads(line)
            qid = str(obj["_id"])
            queries[qid] = obj
    return queries


def load_qrels(file_path: str) -> dict[str, dict[str, int]]:
    """
    TODO

    Load relevance judgments from TSV file.
    Returns dictionary mapping query IDs to candidate relevance scores.
    """
    qrels = defaultdict(dict)
    with open(file_path, "r") as f:
        lines = f.readlines()
        for line in lines[1:]:
            qid, docid, score = line.strip().split("\t")
            qrels[qid][docid] = int(score)
    return qrels


print("Loading dataset...")
corpus = load_corpus("./data/corpus.jsonl")
queries = load_queries("./data/queries.jsonl")
qrels_valid = load_qrels("./data/valid.tsv")


print(f"Loaded {len(corpus)} documents in corpus")
print(f"Loaded {len(queries)} queries")
print(f"Loaded relevance for {len(qrels_valid)} queries (dataset)")

Loading dataset...
Loaded 25657 documents in corpus
Loaded 1000 queries
Loaded relevance for 700 queries (dataset)


## Mise en place d'un graphe

In [6]:
import networkx


def create_corpus_graph(corpus: dict[str, dict]) -> networkx.DiGraph:
    """
    Create a graph from the corpus where each document is a node
    and edges represent citations between documents.
    """
    G = networkx.DiGraph()
    for _id, docdata in corpus.items():
        G.add_node(_id, **docdata)
        metadata = docdata.get("metadata", {})
        citations = metadata.get("cited_by", [])
        for cited_docid in citations:
            if cited_docid in corpus:
                G.add_edge(_id, cited_docid)
    return G


corpus_graph = create_corpus_graph(corpus)
print(
    f"Corpus graph has {corpus_graph.number_of_nodes()} nodes and {corpus_graph.number_of_edges()} edges."
)

Corpus graph has 25657 nodes and 54005 edges.


In [ ]:
num_nodes = corpus_graph.number_of_nodes()
num_edges = corpus_graph.number_of_edges()
print(f"\nNombre de nœuds: {num_nodes}")
print(f"Nombre d'arcs: {num_edges}")

# Densité du graphe
density = networkx.density(corpus_graph)
print(f"\nDensité du graphe: {density:.6f}")

# Degrés entrants et sortants
in_degrees = [corpus_graph.in_degree(node) for node in corpus_graph.nodes()]
out_degrees = [corpus_graph.out_degree(node) for node in corpus_graph.nodes()]

print(f"\n--- Degrés entrants (In-degree) ---")
print(f"Moyenne: {np.mean(in_degrees):.4f}")
print(f"Variance: {np.var(in_degrees):.4f}")
print(f"Écart-type: {np.std(in_degrees):.4f}")
print(f"Min: {np.min(in_degrees)}, Max: {np.max(in_degrees)}")

print(f"\n--- Degrés sortants (Out-degree) ---")
print(f"Moyenne: {np.mean(out_degrees):.4f}")
print(f"Variance: {np.var(out_degrees):.4f}")
print(f"Écart-type: {np.std(out_degrees):.4f}")
print(f"Min: {np.min(out_degrees)}, Max: {np.max(out_degrees)}")

print("\n" + "=" * 60)


Nombre de nœuds: 25657
Nombre d'arcs: 54005

Densité du graphe: 0.000082

--- Degrés entrants (In-degree) ---
Moyenne: 2.1049
Variance: 10.3450
Écart-type: 3.2164
Min: 0, Max: 83

--- Degrés sortants (Out-degree) ---
Moyenne: 2.1049
Variance: 138.6369
Écart-type: 11.7744
Min: 0, Max: 758



## Mise en place d'indicateurs de centralité

In [7]:
def pageRank_clustering(G: networkx.DiGraph, alpha: float = 0.85) -> dict[str, float]:
    """
    Compute PageRank scores for each node in the graph G.
    Returns a dictionary mapping node IDs to their PageRank scores.
    """
    pagerank_scores = networkx.pagerank(G, alpha=alpha)
    return pagerank_scores


def eigenvector_centrality_clustering(
    G: networkx.DiGraph, max_iter: int = 1000, tol: float = 1.0e-6
) -> dict[str, float]:
    """
    Compute Eigenvector Centrality for each node in the graph G.
    Returns a dictionary mapping node IDs to their Eigenvector Centrality scores.
    """
    eigenvector_scores = networkx.eigenvector_centrality(G, max_iter=max_iter, tol=tol)
    return eigenvector_scores


def clustering_coefficient_clustering(
    G: networkx.DiGraph,
) -> dict[str, float]:
    """
    Compute Clustering Coefficient for each node in the graph G.
    Returns a dictionary mapping node IDs to their Clustering Coefficient scores.
    """
    clustering_scores = networkx.clustering(G.to_undirected())
    return clustering_scores


def closeness_centrality_clustering(
    G: networkx.DiGraph,
) -> dict[str, float]:
    """
    Compute Closeness Centrality for each node in the graph G.
    Returns a dictionary mapping node IDs to their Closeness Centrality scores.
    closeness_scores = networkx.closeness_centrality(G)
    """
    closeness_scores = networkx.closeness_centrality(G)
    return closeness_scores


def display_top_score(
    type_clustering: str, corpus_graph: networkx.DiGraph, top_n: int = 10
):
    """
    Display the top N nodes with the highest PageRank scores.
    """
    if type_clustering == "pagerank":
        scores = pageRank_clustering(corpus_graph, alpha=0.85)
    elif type_clustering == "eigenvector":
        scores = eigenvector_centrality_clustering(corpus_graph)
    elif type_clustering == "clustering":
        scores = clustering_coefficient_clustering(corpus_graph)
    elif type_clustering == "closeness":
        scores = closeness_centrality_clustering(corpus_graph)
    else:
        raise ValueError(
            "Invalid clustering type. Choose from 'pagerank', 'eigenvector', 'clustering'."
        )
    sorted_scores = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    print(f"Top {top_n} documents by {type_clustering} score:")
    for rank, (docid, score) in enumerate(sorted_scores[:top_n], start=1):
        print(f"{rank}. Document ID: {docid}, {type_clustering} score: {score:.6f}")
    return sorted_scores


pagerank_scores = display_top_score("pagerank", corpus_graph, top_n=10)
eigenvector_scores = display_top_score("eigenvector", corpus_graph, top_n=10)
clustering_scores = display_top_score("clustering", corpus_graph, top_n=10)
closeness_scores = display_top_score("closeness", corpus_graph, top_n=10)

Top 10 documents by pagerank score:
1. Document ID: bf9db8ca2dce7386cbed1ae0fd6465148cdb2b98, pagerank score: 0.000535
2. Document ID: 4ea92f2ddbb7ce610c6e80377e61397bad7309ff, pagerank score: 0.000534
3. Document ID: 126df9f24e29feee6e49e135da102fbbd9154a48, pagerank score: 0.000456
4. Document ID: 9878cc39383560752c5379a7e9641fc82a4daf7f, pagerank score: 0.000354
5. Document ID: 27a918a368e971a15b545abf63353e269a8ce8a2, pagerank score: 0.000326
6. Document ID: 0c9c3f948eda8fb339e76c6612ca4bd36244efd7, pagerank score: 0.000310
7. Document ID: 9a7def005efb5b4984886c8a07ec4d80152602ab, pagerank score: 0.000298
8. Document ID: 42dfb714a9faddf3f7047d482b2e0531d884dc67, pagerank score: 0.000286
9. Document ID: 57c72cb88843d44b43192741c7010558bb451394, pagerank score: 0.000284
10. Document ID: 58f7accbe36aadd0ef83fd2746c879079eb816bc, pagerank score: 0.000284
Top 10 documents by eigenvector score:
1. Document ID: 383e3b9e408f57052bbbc430b8e6b60c0e31f7ef, eigenvector score: 0.247765
2. Docum

## Centrality comparisons

In [15]:
def comparsons_centralities_scores(
    page_rank_scores,
    betweenness_scores,
    eigenvector_scores,
    top_k: int = 10,
):
    """
    Compare the top documents from PageRank and Betweenness Centrality scores.
    """
    page_rank_top_docs = {docid for docid, score in page_rank_scores[:top_k]}
    betweenness_top_docs = {docid for docid, score in betweenness_scores[:top_k]}
    eigenvecteor_top_docs = {docid for docid, score in eigenvector_scores[:top_k]}

    common_docs = page_rank_top_docs.intersection(betweenness_top_docs)
    print(f"Documents in both top {top_k} PageRank and Betweenness Centrality:")
    for docid in common_docs:
        print(f"- Document ID: {docid}")

    common_docs_eigen = page_rank_top_docs.intersection(eigenvecteor_top_docs)
    print(f"\nDocuments in both top {top_k} PageRank and Eigenvector Centrality:")
    for docid in common_docs_eigen:
        print(f"- Document ID: {docid}")

    common_docs_all = page_rank_top_docs.intersection(
        betweenness_top_docs, eigenvecteor_top_docs
    )
    print(
        f"\nDocuments in top {top_k} of PageRank, Betweenness Centrality, and Eigenvector Centrality:"
    )
    for docid in common_docs_all:
        print(f"- Document ID: {docid}")


comparsons_centralities_scores(
    pagerank_scores, closeness_scores, eigenvector_scores, 100
)

Documents in both top 100 PageRank and Betweenness Centrality:
- Document ID: 167895bdf0f1ef88acc962e7a6f255ab92769485
- Document ID: bf9db8ca2dce7386cbed1ae0fd6465148cdb2b98
- Document ID: ad10d6dec0b5855c3c3e71d334161647d6d4ed9e
- Document ID: 6bc4b1376ec2812b6d752c4f6bc8d8fd0512db91
- Document ID: 42dfb714a9faddf3f7047d482b2e0531d884dc67
- Document ID: bb9e418469d018be7f5ac2c4b2435ccac50088a3
- Document ID: 9a7def005efb5b4984886c8a07ec4d80152602ab
- Document ID: 244c7d89bcd67b26756123edf44c71fd5f39dea6
- Document ID: 44298a4cf816fe8d55c663337932724407ae772b
- Document ID: 6fba4968f1b39d490bf95fe4030e3d385f167074
- Document ID: a7ab6fe31ee11a6b59e4d0c15de9f81661ef0d58
- Document ID: 9878cc39383560752c5379a7e9641fc82a4daf7f
- Document ID: 0fd6d5727215283bca296fe75852475e7c6c7ca0
- Document ID: 383e3b9e408f57052bbbc430b8e6b60c0e31f7ef
- Document ID: 4ea92f2ddbb7ce610c6e80377e61397bad7309ff
- Document ID: b20a5427d79c660fe55282da2533071629bfc533

Documents in both top 100 PageRank and E

## Ajout de l'embedding avec les graphes

In [ ]:
def get_node_embeddings_from_corpus(
    corpus_ids: list[str], corpus_emb: np.ndarray
) -> dict[str, np.ndarray]:
    """
    Create a mapping from node IDs to their embeddings.
    """
    embeddings = {}
    for i, node_id in enumerate(corpus_ids):
        embeddings[node_id] = corpus_emb[i]
    return embeddings


def aggregate_neighbor_embeddings(
    G: networkx.DiGraph, node_embeddings: dict[str, np.ndarray], weight: float = 0.5
) -> dict[str, np.ndarray]:
    """
    Improve node embeddings by aggregating embeddings from neighboring nodes.

    Args:
        G: NetworkX directed graph
        node_embeddings: Dict mapping node IDs to their embedding vectors
        aggregation: Type of aggregation ('mean', 'weighted_mean')
        weight: Weight factor for neighbor embeddings (0 to 1)

    Returns:
        Dict mapping node IDs to improved embedding vectors
    """
    improved_embeddings = {}

    for node in G.nodes():
        if node not in node_embeddings:
            # Si le nœud n'a pas d'embedding, le garder tel quel
            improved_embeddings[node] = node_embeddings.get(node, np.zeros(384))
            continue

        original_emb = node_embeddings[node].copy()
        neighbor_embeddings = []

        for neighbor in G.successors(node):
            if neighbor in node_embeddings:
                neighbor_embeddings.append(node_embeddings[neighbor])

        for neighbor in G.predecessors(node):
            if neighbor in node_embeddings:
                neighbor_embeddings.append(node_embeddings[neighbor])

        if neighbor_embeddings:
            neighbor_agg = np.mean(neighbor_embeddings, axis=0)
            improved_embeddings[node] = (
                1 - weight
            ) * original_emb + weight * neighbor_agg
        else:
            improved_embeddings[node] = original_emb

    return improved_embeddings


def iterative_node_embedding_aggregation(
    G: networkx.DiGraph,
    node_embeddings: dict[str, np.ndarray],
    iterations: int = 2,
    weight: float = 0.3,
) -> dict[str, np.ndarray]:
    """
    Iteratively improve node embeddings by aggregating neighbor embeddings.
    Multiple iterations allow information to propagate further in the graph.

    Args:
        G: NetworkX directed graph
        node_embeddings: Initial node embeddings
        iterations: Number of aggregation iterations
        weight: Weight factor for neighbor embeddings

    Returns:
        Improved node embeddings after multiple iterations
    """
    current_embeddings = node_embeddings.copy()

    for iteration in range(iterations):
        print(f"Iteration {iteration + 1}/{iterations}...")
        current_embeddings = aggregate_neighbor_embeddings(
            G, current_embeddings, weight=weight
        )

    return current_embeddings


encoder_path = "data/sentence_embeddings_all-MiniLM-L6-v2.pkl"
if os.path.exists(encoder_path):
    with open(encoder_path, "rb") as f:
        encoder = pickle.load(f)
    corpus_ids = [_id for _id in corpus.keys()]
    print(corpus_ids)
    node_embeddings = get_node_embeddings_from_corpus(corpus_ids, encoder)
    print(f"Loaded embeddings for {len(node_embeddings)} nodes")
else:
    print(f"Encoder file {encoder_path} not found. Please run the encoder first.")
    raise FileNotFoundError(f"Encoder file {encoder_path} not found.")

improved_embeddings = aggregate_neighbor_embeddings(
    corpus_graph, node_embeddings, weight=0.5
)

improved_embeddings_multi = iterative_node_embedding_aggregation(
    corpus_graph, node_embeddings, iterations=3, weight=0.3
)
print(f"Improved {len(improved_embeddings_multi)} node embeddings")

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
def fetch_query_data(
    query_id: str,
    qrels: dict[str, dict[str, int]],
    queries: dict[str, dict],
    corpus: dict[str, dict],
) -> dict:
    """
    Fetch query text and associated metadata given a query ID.
    """
    if query_id not in queries:
        raise ValueError(f"Query ID {query_id} not found in queries.")
    linked_documents = qrels.get(query_id, {})
    query_data = {
        "query": queries[query_id],
        "linked_documents": {"relevant": [], "irrelevant": []},
    }
    for doc_id in linked_documents:
        if doc_id in corpus:
            doc_data = corpus[doc_id]
            if linked_documents[doc_id] == 1:
                query_data["linked_documents"]["relevant"].append(doc_data)
            else:
                query_data["linked_documents"]["irrelevant"].append(doc_data)
    return query_data


first_id = list(queries.keys())[0]

query_data = fetch_query_data(first_id, qrels_valid, queries, corpus)


def node_search_engine(
    query: str,
    corpus: dict[str, dict],
    embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5,
) -> list[tuple[str, float]]:
    """
    A search engine that retrieves the top_k most similar documents to the query
    based on cosine similarity of sentence embeddings.
    """
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    results = [(list(corpus.keys())[i], similarities[i]) for i in top_indices]
    return results


model = SentenceTransformer("all-MiniLM-L6-v2")

emb = (np.array([improved_embeddings_multi[docid] for docid in corpus.keys()]),)

results = node_search_engine(
    query_data["query"]["text"],
    corpus,
    np.array([improved_embeddings_multi[docid] for docid in corpus.keys()]),
    model,
    top_k=5,
)

No sentence-transformers model found with name sentence-transformers/e5-small-v2. Creating a new one with mean pooling.


OSError: sentence-transformers/e5-small-v2 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [19]:
print("Top 10 search results (Sentence Embeddings):")
for doc_id, score in results:
    print(
        f"Document ID: {doc_id}, Similarity Score: {score:.4f}, Title: {corpus[doc_id]['title']}"
    )
relevant_article_count = sum(
    1
    for doc_id, score in results
    if doc_id in [doc["_id"] for doc in query_data["linked_documents"]["relevant"]]
)
print(f"Number of relevant articles in top 10 results: {relevant_article_count}")

score = 0
for i, query in enumerate(queries.values()):
    results = node_search_engine(
        query["text"],
        corpus,
        np.array([improved_embeddings_multi[docid] for docid in corpus.keys()]),
        model,
        top_k=10,
    )
    relevant_article_count = sum(
        1
        for doc_id, score in results
        if doc_id in qrels_valid.get(str(query["_id"]), {})
        and qrels_valid[str(query["_id"])][doc_id] == 1
    )
    # print(f"Query ID: {query['_id']}, Number of relevant articles in top 10 results: {relevant_article_count}")
    score += relevant_article_count
    if i % 100 == 0:
        print(f"Processed {i} queries...")
print(f"Total score across all queries: {score}")
print(f"Total score across all queries: {score / (len(queries) * 10)}")

Top 10 search results (Sentence Embeddings):
Document ID: 776584e054bd8ba1ff6c4906eb947fc0abb0abc3, Similarity Score: 0.1060, Title: Efficient aircraft spare parts inventory management under demand uncertainty
Document ID: 4ceac711f4b158503b11434f082dbe31014d4a9a, Similarity Score: 0.1044, Title: On-Line Economic Optimization of Energy Systems Using Weather Forecast Information
Document ID: f2d92b79bfad01cfada381653fca76e142357252, Similarity Score: 0.0984, Title: Process reengineering in the public sector : learning some private sector lessons
Document ID: cd31ecb3b58d1ec0d8b6e196bddb71dd6a921b6d, Similarity Score: 0.0962, Title: Economic dispatch for a microgrid considering renewable energy cost functions
Document ID: 768b18d745639fcfb157fe16cbd957ca60ebfc2e, Similarity Score: 0.0958, Title: Design of Electronic Throttle Valve Position Control System using Nonlinear PID Controller
Number of relevant articles in top 10 results: 0
Processed 0 queries...
Processed 100 queries...
Process

KeyboardInterrupt: 